# Solutions: Specification Tests

These are the solutions for Exercises 1-3 from Notebook 03: Specification Tests.

In [1]:
%matplotlib inline

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


import panelbox
from panelbox.diagnostics.hausman import hausman_test
from panelbox.diagnostics.specification import j_test
from panelbox.models.static import FixedEffects, RandomEffects

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100
np.random.seed(42)

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "specification"
OUTPUT_DIR = BASE_DIR / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures" / "specification"
TABLES_DIR = OUTPUT_DIR / "tables" / "specification"

for d in [FIGURES_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Setup complete!")
print(f"PanelBox version: {panelbox.__version__}")

Setup complete!
PanelBox version: 1.0.0


---

## Exercise 1 Solution: Basic Hausman Test

**Task**: Perform a Hausman test on the NLS wage data to decide between FE and RE.

In [2]:
# Step 1: Load data
nls = pd.read_csv(DATA_DIR / "nlswork.csv")
print(
    f"NLS data: {nls.shape[0]} observations, {nls['idcode'].nunique()} workers, {nls['year'].nunique()} years"
)
display(nls.head())

NLS data: 7500 observations, 500 workers, 15 years


,idcode,year,ln_wage,experience,tenure,education,union,married,hours,industry
0,1,1968,1.5218,3.56,1.38,12,0,1,42.0,11
1,1,1970,2.4459,6.08,5.60,12,1,0,47.3,11
2,1,1972,1.5970,5.56,0.50,12,0,1,41.1,11
3,1,1974,1.8724,9.13,1.34,12,0,0,38.5,11
4,1,1976,1.7113,10.59,1.41,12,1,1,43.9,11


In [3]:
# Step 2: Estimate FE model (time-varying variables only)
fe_model = FixedEffects(
    "ln_wage ~ experience + tenure + union + married",
    data=nls,
    entity_col="idcode",
    time_col="year",
)
fe_result = fe_model.fit()

print("Fixed Effects Results:")
print(fe_result.summary())

Fixed Effects Results:
                       Fixed Effects Estimation Results                       
Formula: ln_wage ~ experience + tenure + union + married
Model:   Fixed Effects
------------------------------------------------------------------------------
No. Observations:               7,500
No. Entities:                     500
No. Time Periods:                  15
Degrees of Freedom:             6,996
R-squared:                     0.1987
Adj. R-squared:                0.1411
R-squared (within):            0.1987
R-squared (between):           1.0000
R-squared (overall):           0.7323
Standard Errors:            nonrobust
F-statistic (FE vs OLS):      31.4681
F-test p-value:                0.0000
Variable        Coef.        Std.Err.     t        P>|t|    [0.025     0.975]    
------------------------------------------------------------------------------
experience           0.0198      0.0005  36.929  0.0000    0.0188    0.0209 ***
tenure               0.0158      0.0018   

In [4]:
# Step 3: Estimate RE model (same variables as FE for fair comparison)
re_model = RandomEffects(
    "ln_wage ~ experience + tenure + union + married",
    data=nls,
    entity_col="idcode",
    time_col="year",
)
re_result = re_model.fit()

print("Random Effects Results:")
print(re_result.summary())

Random Effects Results:
                   Random Effects (GLS) Estimation Results                    
Formula: ln_wage ~ experience + tenure + union + married
Model:   Random Effects (GLS)
------------------------------------------------------------------------------
No. Observations:               7,500
No. Entities:                     500
No. Time Periods:                  15
Degrees of Freedom:             7,495
R-squared:                     0.1082
Adj. R-squared:                0.1077
R-squared (within):            0.2050
R-squared (between):           0.0629
R-squared (overall):           0.1082
Standard Errors:            nonrobust
Variable        Coef.        Std.Err.     t        P>|t|    [0.025     0.975]    
------------------------------------------------------------------------------
Intercept            1.4422      0.0154  93.870  0.0000    1.4121    1.4723 ***
experience           0.0202      0.0005  37.083  0.0000    0.0192    0.0213 ***
tenure               0.0171   

In [5]:
# Step 4: Run Hausman test
hausman_result = hausman_test(fe_result, re_result)
hausman_result.summary()


Hausman Specification Test
H0: Random Effects estimator is consistent and efficient
H1: Random Effects estimator is inconsistent

Test statistic: 38.8019
P-value: 0.0000
Degrees of freedom: 4
Method: chi2

----------------------------------------
Interpretation:
Reject H0: Random Effects estimator appears inconsistent (p=0.0000).
Fixed Effects is preferred. This suggests correlation between
unobserved heterogeneity and regressors.

Strong evidence against Random Effects (p < 0.01)

Variables compared: experience, married, tenure, union


In [6]:
# Step 5: Detailed coefficient comparison
common_vars = hausman_result.common_vars
comparison = pd.DataFrame(
    {
        "FE": fe_result.params[common_vars],
        "RE": re_result.params[common_vars],
        "Difference": fe_result.params[common_vars] - re_result.params[common_vars],
    }
)
comparison["% Diff"] = (comparison["Difference"] / comparison["FE"] * 100).round(1)

print("Coefficient Comparison")
print("=" * 60)
display(comparison.round(6))

Coefficient Comparison


,FE,RE,Difference,% Diff
experience,0.019809,0.020247,-0.000437,-2.2
married,0.039921,0.040368,-0.000448,-1.1
tenure,0.015767,0.017062,-0.001295,-8.2
union,0.103407,0.094548,0.008859,8.6


In [7]:
# Step 6: Interpretation
print("Interpretation")
print("=" * 60)

if hausman_result.pvalue < 0.05:
    print(f"Hausman test REJECTS H0 (p = {hausman_result.pvalue:.4f})")
    print()
    print("→ Use Fixed Effects. The Random Effects estimator is inconsistent.")
    print()
    print("Economic interpretation:")
    print("  The unobserved individual effect (ability) is correlated with")
    print("  the time-varying regressors (experience, tenure, union, married).")
    print("  This makes economic sense: more able workers accumulate more")
    print("  experience and may have longer tenure.")
    print()
    print("Implication for education:")
    print("  - FE cannot estimate the education coefficient (time-invariant)")
    print("  - The RE education coefficient is likely biased upward because")
    print("    ability is positively correlated with both education and wages")
else:
    print(f"Hausman test FAILS TO REJECT H0 (p = {hausman_result.pvalue:.4f})")
    print()
    print("→ Random Effects is acceptable and more efficient.")
    print()
    print("We can also estimate time-invariant effects like education:")

    # Estimate RE with education
    re_edu = RandomEffects(
        "ln_wage ~ experience + tenure + union + married + education",
        data=nls,
        entity_col="idcode",
        time_col="year",
    ).fit()
    print(f"  Education coefficient: {re_edu.params['education']:.4f}")
    print(
        f"  Interpretation: {re_edu.params['education'] * 100:.2f}% wage increase per year of education"
    )
    print()
    print("  However, note that failure to reject does NOT prove RE is correct.")
    print("  The test may simply lack power to detect the endogeneity.")

Interpretation
Hausman test REJECTS H0 (p = 0.0000)

→ Use Fixed Effects. The Random Effects estimator is inconsistent.

Economic interpretation:
  The unobserved individual effect (ability) is correlated with
  the time-varying regressors (experience, tenure, union, married).
  This makes economic sense: more able workers accumulate more
  experience and may have longer tenure.

Implication for education:
  - FE cannot estimate the education coefficient (time-invariant)
  - The RE education coefficient is likely biased upward because
    ability is positively correlated with both education and wages


---

## Exercise 2 Solution: J-Test Application

**Task**: Compare three production function specifications using the J-test.

In [8]:
# Step 1: Load data and estimate all three models
firm = pd.read_csv(DATA_DIR / "firm_productivity.csv")
print(
    f"Firm data: {firm.shape[0]} obs, {firm['firm_id'].nunique()} firms, {firm['year'].nunique()} years"
)

# Model A: Basic Cobb-Douglas
result_a = FixedEffects(
    "log_output ~ log_capital + log_labor", data=firm, entity_col="firm_id", time_col="year"
).fit()

# Model B: With Materials
result_b = FixedEffects(
    "log_output ~ log_capital + log_labor + log_materials",
    data=firm,
    entity_col="firm_id",
    time_col="year",
).fit()

# Model C: Full specification
result_c = FixedEffects(
    "log_output ~ log_capital + log_labor + log_materials + rd_intensity",
    data=firm,
    entity_col="firm_id",
    time_col="year",
).fit()

# Compare coefficients
comp = pd.DataFrame(
    {
        "A: Basic": result_a.params,
        "B: Materials": result_b.params,
        "C: Full": result_c.params,
    }
).fillna("")

print("\nProduction Function Estimates (FE)")
print("=" * 60)
display(comp.round(4))
print(
    f"\nWithin R²: A={result_a.rsquared_within:.4f}, B={result_b.rsquared_within:.4f}, C={result_c.rsquared_within:.4f}"
)

Firm data: 4000 obs, 200 firms, 20 years



Production Function Estimates (FE)


,A: Basic,B: Materials,C: Full
log_capital,0.29884,0.298806,0.2989
log_labor,0.356281,0.349457,0.3494
log_materials,,0.249676,0.2496
rd_intensity,,,0.4503



Within R²: A=0.7249, B=0.9430, C=0.9434


In [9]:
# Step 2: J-test A vs B
print("J-Test: Model A vs Model B")
print("=" * 60)
jtest_ab = j_test(result_a, result_b, model1_name="Basic (K,L)", model2_name="Materials (K,L,M)")
print(jtest_ab.interpretation())
print()
display(jtest_ab.summary().round(4))

J-Test: Model A vs Model B
J-Test Results (Both Directions):

Forward Test (Basic (K,L) vs Materials (K,L,M)):
  H0: Basic (K,L) is correct
  p-value = 0.0000
  Result: REJECT H0

Reverse Test (Materials (K,L,M) vs Basic (K,L)):
  H0: Materials (K,L,M) is correct
  p-value = 0.0000
  Result: REJECT H0

Overall Interpretation:
------------------------------------------------------------
BOTH MODELS REJECTED
  - Neither model adequately encompasses the other
  - Both models have explanatory power the other lacks
  - Consider a more comprehensive specification




,Test,Null Hypothesis,Test Statistic,p-value,Coefficient,Std. Error
0,Forward,"Basic (K,L) is correct",256.7689,0.0,1.0000,0.0039
1,Reverse,"Materials (K,L,M) is correct",123.8651,0.0,0.8434,0.0068


In [10]:
# Step 3: J-test B vs C
print("J-Test: Model B vs Model C")
print("=" * 60)
jtest_bc = j_test(
    result_b, result_c, model1_name="Materials (K,L,M)", model2_name="Full (K,L,M,R&D)"
)
print(jtest_bc.interpretation())
print()
display(jtest_bc.summary().round(4))

J-Test: Model B vs Model C
J-Test Results (Both Directions):

Forward Test (Materials (K,L,M) vs Full (K,L,M,R&D)):
  H0: Materials (K,L,M) is correct
  p-value = 0.0000
  Result: REJECT H0

Reverse Test (Full (K,L,M,R&D) vs Materials (K,L,M)):
  H0: Full (K,L,M,R&D) is correct
  p-value = 0.0000
  Result: REJECT H0

Overall Interpretation:
------------------------------------------------------------
BOTH MODELS REJECTED
  - Neither model adequately encompasses the other
  - Both models have explanatory power the other lacks
  - Consider a more comprehensive specification




,Test,Null Hypothesis,Test Statistic,p-value,Coefficient,Std. Error
0,Forward,"Materials (K,L,M) is correct",192.6784,0.0,1.0000,0.0052
1,Reverse,"Full (K,L,M,R&D) is correct",188.4383,0.0,0.9954,0.0053


In [11]:
# Step 4: Final specification choice
print("Final Specification Decision")
print("=" * 60)


# Classify outcomes
def classify_jtest(jt):
    fwd_reject = jt.forward["pvalue"] < 0.05
    rev_reject = jt.reverse["pvalue"] < 0.05
    if fwd_reject and not rev_reject:
        return "Prefer Model 2"
    elif not fwd_reject and rev_reject:
        return "Prefer Model 1"
    elif fwd_reject and rev_reject:
        return "Both rejected"
    else:
        return "Both acceptable"


outcome_ab = classify_jtest(jtest_ab)
outcome_bc = classify_jtest(jtest_bc)

print(f"  A vs B: {outcome_ab}")
print(f"  B vs C: {outcome_bc}")
print()

# Decision logic
if "Prefer Model 2" in outcome_ab or "Both rejected" in outcome_ab:
    if "Prefer Model 2" in outcome_bc or "Both rejected" in outcome_bc:
        print("Recommendation: Use Model C (Full specification)")
        print("  Materials and R&D both contribute to explaining output")
    else:
        print("Recommendation: Use Model B (With Materials)")
        print("  Materials matter but R&D does not add significant information")
else:
    print("Recommendation: Use Model A (Basic Cobb-Douglas)")
    print("  The simpler specification is adequate")

Final Specification Decision
  A vs B: Both rejected
  B vs C: Both rejected

Recommendation: Use Model C (Full specification)
  Materials and R&D both contribute to explaining output


---

## Exercise 3 Solution: Singular Hausman Matrix

**Task**: Demonstrate and handle the case where the Hausman test produces a negative statistic due to a non-positive-definite variance difference matrix.

In [12]:
# Step 1: Simulate data with very small within-variation
np.random.seed(42)
N_sim, T_sim = 30, 5

records = []
for i in range(1, N_sim + 1):
    alpha = np.random.normal(0, 2.0)  # Large fixed effect variance
    x1_mean = np.random.normal(5, 2)  # Different mean per entity

    for t in range(T_sim):
        # x1 has VERY small within-variation (mostly between-variation)
        x1 = x1_mean + np.random.normal(0, 0.05)  # tiny within-variance!
        x2 = np.random.normal(3, 1)  # normal within-variation

        y = 1 + 0.5 * x1 + 0.3 * x2 + alpha + np.random.normal(0, 0.5)

        records.append(
            {"entity": i, "time": t, "y": round(y, 4), "x1": round(x1, 4), "x2": round(x2, 4)}
        )

df_singular = pd.DataFrame(records)

# Check within-variation
within_var = df_singular.groupby("entity")["x1"].var().mean()
between_var = df_singular.groupby("entity")["x1"].mean().var()
print(f"x1 within-variance:  {within_var:.6f}")
print(f"x1 between-variance: {between_var:.6f}")
print(f"Ratio (within/between): {within_var / between_var:.6f}")
print("→ Almost all variation is BETWEEN entities, not within!")

x1 within-variance:  0.002699
x1 between-variance: 2.991510
Ratio (within/between): 0.000902
→ Almost all variation is BETWEEN entities, not within!


In [13]:
# Step 2: Attempt Hausman test
fe_sing = FixedEffects("y ~ x1 + x2", df_singular, "entity", "time").fit()
re_sing = RandomEffects("y ~ x1 + x2", df_singular, "entity", "time").fit()

print("FE estimates:", fe_sing.params.to_dict())
print("RE estimates:", {k: v for k, v in re_sing.params.items() if k != "Intercept"})

# Run Hausman test and catch warnings
import warnings

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    h_sing = hausman_test(fe_sing, re_sing)

    print(f"\nHausman statistic: {h_sing.statistic:.4f}")
    print(f"P-value: {h_sing.pvalue:.4f}")

    if w:
        print(f"\nWarnings ({len(w)}):")
        for warning in w:
            print(f"  ⚠ {warning.message}")

FE estimates: {'x1': 0.2747218994126273, 'x2': 0.3377576114771149}
RE estimates: {'x1': -0.01913906358784448, 'x2': 0.34213503949714497}

Hausman statistic: -1.5854
P-value: 1.0000

Warnings (1):
  ⚠ Variance difference matrix is not positive semi-definite. This may indicate model misspecification. Minimum eigenvalue: -7.93e-06


In [14]:
# Step 3: Examine eigenvalues of V_FE - V_RE
common = h_sing.common_vars
V_fe = fe_sing.cov_params.loc[common, common].values
V_re = re_sing.cov_params.loc[common, common].values
V_diff = V_fe - V_re

eigs = np.linalg.eigvalsh(V_diff)

print("Eigenvalue Analysis of V_FE - V_RE")
print("=" * 50)
for i, e in enumerate(eigs):
    status = "✓ positive" if e > 1e-10 else ("≈ zero" if abs(e) < 1e-10 else "✗ NEGATIVE")
    print(f"  λ_{i + 1} = {e:.2e}  {status}")

if any(e < -1e-8 for e in eigs):
    print("\n→ Matrix is NOT positive semi-definite!")
    print("  The Hausman statistic may be negative or unreliable.")
    print("\n  This happens because V(β_FE) is not necessarily larger than")
    print("  V(β_RE) in finite samples, especially when within-variation is small.")
else:
    print("\n→ Matrix is positive semi-definite (test is valid)")

Eigenvalue Analysis of V_FE - V_RE
  λ_1 = -7.93e-06  ✗ NEGATIVE
  λ_2 = 6.13e-01  ✓ positive

→ Matrix is NOT positive semi-definite!
  The Hausman statistic may be negative or unreliable.

  This happens because V(β_FE) is not necessarily larger than
  V(β_RE) in finite samples, especially when within-variation is small.


In [15]:
# Step 4: Discussion of alternatives
print("Alternatives When Hausman Test Fails")
print("=" * 50)
print()
print("1. Use a larger sample (increase N or T)")
print("   More data makes the asymptotic theory more reliable.")
print()
print("2. Use robust Hausman test")
print("   Cluster-robust standard errors can help in some cases.")
print()
print("3. Use the Mundlak approach")
print("   Add group means of X to the RE model as additional regressors.")
print("   Test whether the group means are jointly significant.")
print()
print("4. Use a manual Wald test")
print("   Test coefficient equality directly rather than using the")
print("   standard Hausman formula.")
print()
print("5. Consider the economic context")
print("   If theory strongly suggests endogeneity, use FE regardless")
print("   of what the test says.")
print()

# Quick demo: increase sample size
print("\nDemo: Effect of sample size on Hausman test")
print("-" * 50)
for N_test in [30, 100, 300]:
    np.random.seed(42)
    recs = []
    for i in range(1, N_test + 1):
        alpha = np.random.normal(0, 1.0)
        x_mean = np.random.normal(5, 2)
        for t in range(10):
            x1 = x_mean + np.random.normal(0, 0.1)
            x2 = np.random.normal(3, 1)
            y = 1 + 0.5 * x1 + 0.3 * x2 + alpha + np.random.normal(0, 0.5)
            recs.append({"entity": i, "time": t, "y": y, "x1": x1, "x2": x2})

    df_test = pd.DataFrame(recs)
    try:
        fe_t = FixedEffects("y ~ x1 + x2", df_test, "entity", "time").fit()
        re_t = RandomEffects("y ~ x1 + x2", df_test, "entity", "time").fit()
        h_t = hausman_test(fe_t, re_t)

        V_d = (
            fe_t.cov_params.values
            - re_t.cov_params.loc[fe_t.params.index, fe_t.params.index].values
        )
        min_eig = np.linalg.eigvalsh(V_d).min()
        print(f"  N={N_test:3d}: H={h_t.statistic:8.3f}, p={h_t.pvalue:.4f}, min_eig={min_eig:.2e}")
    except Exception as e:
        print(f"  N={N_test:3d}: Error - {e}")

Alternatives When Hausman Test Fails

1. Use a larger sample (increase N or T)
   More data makes the asymptotic theory more reliable.

2. Use robust Hausman test
   Cluster-robust standard errors can help in some cases.

3. Use the Mundlak approach
   Add group means of X to the RE model as additional regressors.
   Test whether the group means are jointly significant.

4. Use a manual Wald test
   Test coefficient equality directly rather than using the
   standard Hausman formula.

5. Consider the economic context
   If theory strongly suggests endogeneity, use FE regardless
   of what the test says.


Demo: Effect of sample size on Hausman test
--------------------------------------------------
  N= 30: H=   1.671, p=0.4336, min_eig=1.53e-05
  N=100: H=   0.369, p=0.8314, min_eig=3.56e-06


  N=300: H=   0.561, p=0.7554, min_eig=1.24e-06


---

*End of solutions for Notebook 03: Specification Tests*

**Note**: Exercise 4 is independent and has no provided solution. Use the tools and techniques from the main notebook to conduct your own specification analysis.